# Loan Approval Prediction Using Machine Learning
**AIA Project | Google Colab**

## Objective
Build a supervised machine-learning classification model to predict whether a loan application is approved (`Y`) or rejected (`N`) using applicant information.

**Dataset:** Loan Prediction dataset (614 records, 13 columns). This is the commonly used public dataset; it is not a 1,000+ row dataset.

**Algorithms compared:** Logistic Regression, Decision Tree, and Random Forest.

> Educational project only. Predictions are not actual bank decisions or financial advice.

## 1. Import libraries
We use Pandas and NumPy for data handling, Matplotlib/Seaborn for visualization, and Scikit-learn for preprocessing, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")

## 2. Upload and load the dataset
Download the CSV file from the dataset source linked in `README.md`. Upload the CSV file when the file picker opens.

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

# Standardize column names by removing extra spaces
df.columns = df.columns.str.strip()

print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the dataset
`head()` shows the first five records. `info()` shows column types and non-null counts. `describe()` summarizes numerical columns.

In [ ]:
df.info()
display(df.describe(include="all").T)
print("Missing values in each column:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

## 4. Data cleaning
Remove duplicate rows. Convert `Dependents` values such as `3+` to `3`. Missing values will be imputed inside the preprocessing pipeline: median for numerical columns and most frequent value for categorical columns. This keeps preprocessing learned from training data only.

In [ ]:
df = df.drop_duplicates().copy()

if "Dependents" in df.columns:
    df["Dependents"] = df["Dependents"].replace("3+", "3")
    df["Dependents"] = pd.to_numeric(df["Dependents"], errors="coerce")

print("Shape after duplicate removal:", df.shape)
print("Remaining missing values (handled later by imputers):")
display(df.isnull().sum())

## 5. Exploratory data analysis
Check the target distribution and numeric feature correlations. Correlation describes linear association; it does not prove cause and effect.

In [ ]:
target_col = "Loan_Status"
if target_col not in df.columns:
    raise ValueError("The dataset must contain a Loan_Status column.")

sns.countplot(data=df, x=target_col)
plt.title("Loan Status Distribution")
plt.xlabel("Loan Status (Y = Approved, N = Rejected)")
plt.show()

numeric_for_corr = df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 6))
sns.heatmap(numeric_for_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Features")
plt.show()

## 6. Prepare features and target
`Loan_ID` is an identifier, so it is removed from model inputs. `Loan_Status` is the target. The remaining columns are features.

In [ ]:
X = df.drop(columns=[target_col, "Loan_ID"], errors="ignore")
y = df[target_col].astype(str).str.strip()

# Normalize common target labels
y = y.replace({"Y": 1, "N": 0, "Approved": 1, "Rejected": 0})
y = pd.to_numeric(y, errors="raise").astype(int)

# Identify numeric and categorical feature columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target counts:")
print(y.value_counts())

## 7. Split the data
Use 80% of the records for training and 20% for testing. `stratify=y` keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

## 8. Preprocess and train models
Numerical columns: missing values are filled with the median and values are standardized.
Categorical columns: missing values are filled with the most frequent value and categories are one-hot encoded.

A pipeline ensures that the imputer, scaler, and encoder are fitted only on the training data.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6,
                                             random_state=42, class_weight="balanced")
}

trained_models = {}
results = []

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    trained_models[name] = pipeline
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(results_df)

## 9. Evaluate the best model
We select the model with the highest test F1-score. Accuracy is the overall proportion of correct predictions. Precision measures how many predicted approvals were correct; recall measures how many actual approvals were found. F1-score balances precision and recall.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Selected model:", best_model_name)
print("\nClassification report:")
print(classification_report(y_test, best_predictions,
                            target_names=["Rejected", "Approved"],
                            zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    display_labels=["Rejected", "Approved"],
    cmap="Blues"
)
plt.title("Confusion Matrix - " + best_model_name)
plt.show()

## 10. Save the trained model
The selected pipeline includes preprocessing and the trained classifier. Saving the complete pipeline means future inputs receive the same preprocessing.

In [ ]:
model_file = "loan_approval_model.pkl"
joblib.dump(best_model, model_file)
print("Saved:", model_file)

# Download the trained model from Colab
files.download(model_file)

## 11. Try a sample prediction
This sample uses the feature columns from the dataset. Values are illustrative only. Change them to test another applicant profile.

In [ ]:
sample_applicant = X_test.iloc[[0]].copy()
sample_prediction = best_model.predict(sample_applicant)[0]

print("Sample applicant details:")
display(sample_applicant)
print("Prediction:", "Approved" if sample_prediction == 1 else "Rejected")

## 12. Conclusion
This project demonstrates the complete machine-learning workflow: dataset inspection, duplicate removal, missing-value handling, categorical encoding, correlation analysis, an 80:20 train-test split, model comparison, evaluation, and model saving.

The best model is selected using the test F1-score from the three algorithms. Results depend on the dataset and split. This model is for academic demonstration and should not be used to make real lending decisions.